# AdaptCLIP ViSA Experiment

ViSA 데이터셋에서 `models.adaptclip.AdaptCLIP`을 기존 heatmap 노트북과 같은 흐름으로 실행합니다. Heatmap은 anomaly split에서 시각화하고, AUROC는 normal/anomaly split을 함께 계산합니다.


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "models").is_dir() and (path / "datasets").is_dir() and (path / "utils").is_dir():
            return path
    raise RuntimeError("Repository root was not found from the current working directory.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) in sys.path:
    sys.path.remove(str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT))

print("repo root:", REPO_ROOT)


In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

import config_visa
from datasets.visa import ViSA
from models.adaptclip import AdaptCLIP
from utils.visualization import show_result


In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(device)


In [ ]:
train_data = ViSA(
    config_visa.CLASS_NAME,
    phase="Normal",
    batch_size=config_visa.BATCH_SIZE,
    shuffle=False,
    limit=config_visa.TRAIN_LIMIT,
)

test_normal = ViSA(
    config_visa.CLASS_NAME,
    phase="Normal",
    batch_size=config_visa.BATCH_SIZE,
    shuffle=False,
    limit=config_visa.TEST_LIMIT,
)

test_anomaly = ViSA(
    config_visa.CLASS_NAME,
    phase="Anomaly",
    batch_size=config_visa.BATCH_SIZE,
    shuffle=False,
    limit=config_visa.TEST_LIMIT,
)

test_data = test_anomaly

print("class:", config_visa.CLASS_NAME)
print("train:", len(train_data), "normal test:", len(test_normal), "anomaly test:", len(test_anomaly))


In [ ]:
adaptclip = AdaptCLIP(
    category=config_visa.CLASS_NAME,
    device=device,
    checkpoint_path="visa",
    out_size_h=256,
    out_size_w=256,
    img_resize=336,
    img_cropsize=336,
    batch_size=config_visa.BATCH_SIZE,
    use_prompt_query=True,
)

adaptclip.fit(train_data)


In [ ]:
scores = []
heatmaps = []
imgs = []

for img, label in test_data:
    score, heatmap = adaptclip.predict(img)
    scores.append(score.item())
    heatmaps.append(heatmap)
    imgs.append(img)

print("anomaly scores:", np.round(scores, 4).tolist())


In [ ]:
auc_scores = []
auc_labels = []

for dataset in [test_normal, test_anomaly]:
    for img, label in dataset:
        score, _ = adaptclip.predict(img)
        auc_scores.append(score.item())
        auc_labels.append(int(label))

image_auc = roc_auc_score(auc_labels, auc_scores)
print("image AUROC:", image_auc)


In [ ]:
show_result(test_data, scores, heatmaps, k=min(4, len(test_data)))
